# **Removing Duplicate/Inadequate Rows**

In [ ]:
import pandas as pd

df = pd.read_csv("scraped_jobs.csv")
df = df.dropna(subset=["description"])

initial_rows = len(df)
print("Initial dataset size:",initial_rows)

df = df.drop_duplicates(
    subset=["role", "description"],
    keep="first"
)



import re

def has_valid_words(text, min_words=10):
    words = re.findall(r"[a-zA-Z]{2,}", text)
    return len(words) >= min_words

df = df[df["description"].apply(has_valid_words)]

len(df)



df["desc_length"] = df["description"].str.len()
avg_length = df["desc_length"].mean()
MIN_LENGTH = avg_length * 0.4
df = df[df["desc_length"] >= MIN_LENGTH]

def valid_job_title(title):
    if pd.isna(title):
        return False
    title = title.lower()
    if len(title) < 3:
        return False
    if not re.search(r"[a-zA-Z]", title):
        return False
    return True

df = df[df["role"].apply(valid_job_title)]

print(f"Final dataset size: {len(df)} job postings")



Initial Dataset Size: 954
Final dataset size: 764 job postings


# **Tokenization & Lemmatization**

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)      # remove html
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_description"] = df["description"].apply(clean_text)

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc 
            if not token.is_stop and token.is_alpha]

df["tokens"] = df["clean_description"].apply(lemmatize)
